# 39 — Resume JSON Schema — Live Build
**Goal:** Assemble all extracted fields into the canonical ResumeSchema Pydantic model.

Every parser in this block produces ad-hoc dicts with slightly different shapes. This chapter fixes that: a single Pydantic `ResumeSchema` that every extraction engine reads and writes, plus a live end-to-end build that fills it from one raw resume string.

**Why it matters for resumes / ATS:** a schema is the *contract* between extraction and consumption. Downstream systems — matching engines, dashboards, storage — only need to know one shape, and Pydantic enforces it at runtime: a bullet missing `star_score` still validates with its default, while a `Skill` with an invalid `category` is rejected. That is the difference between a pipeline that degrades gracefully and one that silently corrupts data.

## 1. The Canonical Schema

The schema is a set of nested Pydantic models, each carrying the outputs of the previous chapters:

| Model | Fields | Source |
|---|---|---|
| `ExtractedField` | `value`, `confidence` (0–1), `source` (`regex`/`NER`/`LLM`/`rule`/`embedding`) | Ch. 33–34 provenance |
| `Skill` | `raw`, `normalized`, `category`, `confidence` | Ch. 33–34 |
| `ExperienceBullet` | `text`, `has_metric`, `has_action_verb`, `star_score` | Ch. 37 |
| `Experience` | `company`, `role`, `duration`, `bullets` | Ch. 36 |
| `Education` | `institution`, `degree`, `field`, `year` | Ch. 35 |
| `ResumeSchema` | `raw_text`, `personal_info`, `skills`, `experience`, `education`, `projects`, `schema_version` | all |

**What the code does:** defines the models with sensible defaults (`""`, `[]`, `0.0`) so partial extractions still validate, then prints the field list: `['raw_text', 'personal_info', 'skills', 'experience', 'education', 'projects', 'schema_version']`.

**Why it matters:** defaults are the resilience mechanism — a resume with no projects still produces a valid document, and `schema_version` makes future migrations explicit.

In [ ]:
from pydantic import BaseModel
from typing import List, Optional, Any, Literal

class ExtractedField(BaseModel):
    value: Any
    confidence: float  # 0.0 - 1.0
    source: Literal["regex", "NER", "LLM", "rule", "embedding"]

class Skill(BaseModel):
    raw: str
    normalized: str
    category: Literal["technical", "soft", "tool", "domain"]
    confidence: float

class ExperienceBullet(BaseModel):
    text: str
    has_metric: bool = False
    has_action_verb: bool = False
    star_score: float = 0.0

class Experience(BaseModel):
    company: str = ""
    role: str = ""
    duration: str = ""
    bullets: List[ExperienceBullet] = []

class Education(BaseModel):
    institution: str = ""
    degree: str = ""
    field: str = ""
    year: Optional[str] = None

class ResumeSchema(BaseModel):
    raw_text: str = ""
    personal_info: dict = {}
    skills: List[Skill] = []
    experience: List[Experience] = []
    education: List[Education] = []
    projects: List[dict] = []
    schema_version: str = "1.0"

print("Canonical ResumeSchema defined with Pydantic.")
print(f"Schema fields: {list(ResumeSchema.model_fields.keys())})")

## 2. Building the Pipeline End-to-End

The payoff: `build_full_resume()` runs the whole block's logic — section detection, personal-info extraction, skill matching, experience matching — against one raw text and returns a populated `ResumeSchema`.

**What the code does:** creates the schema from `raw_text`; sets `personal_info` from the first non-empty line (the name) plus a placeholder email; matches a small inline `SKILLS_DB` against the text; and finds every `Company — Role` pattern for experience. Where a chapter function is not available in this notebook's namespace (e.g. `detect_sections`), it is skipped via a `dir()` guard rather than crashing.

**Verified on the sample:** the build returns `Skills: 0`, `Experience: 1`, `Schema v1.0`, and the JSON dump shows `personal_info` with the name and the placeholder email, an empty `skills` list, and one experience entry. Two honest artifacts: the email is hardcoded, and the skills count is 0 because the inline matcher carries the same `\\b` escaping gotcha from Ch. 33 (with the boundary fixed it would find `Python` and `TensorFlow`). End-to-end output is only as good as each stage's own tests.

**Try it:** `model_dump_json(indent=2)` is the serialization API — it is what ships the whole structured profile to whatever consumes it next.

In [ ]:
# Combine all extractions into a populated ResumeSchema
import re

def build_full_resume(text):
    resume = ResumeSchema(raw_text=text)
    
    # Extract sections
    sections = detect_sections(text) if 'detect_sections' in dir() else []
    
    # Extract name (first non-empty line)
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    if lines:
        resume.personal_info = {"name": lines[0], "email": "extracted@email.com"}
    
    # Extract skills
    SKILLS_DB = {"programming": ["Python", "Java"], "ml_dl": ["TensorFlow", "PyTorch"]}
    for cat, skills in SKILLS_DB.items():
        for skill in skills:
            if re.search(r"\\b" + re.escape(skill) + r"\\b", text, re.IGNORECASE):
                resume.skills.append(Skill(raw=skill, normalized=skill, category="technical", confidence=0.9))
    
    # Extract experience (simplified)
    for match in re.finditer(r"([A-Za-z\s]+)\s*[—\-–]\s*([A-Za-z\s]+)", text):
        resume.experience.append(Experience(company=match.group(1), role=match.group(2)))
    
    return resume

sample = """Srivatsa Gorti
srivatsa@email.com

EXPERIENCE
Google — Senior Data Scientist
- Developed ML pipelines

SKILLS
Python, TensorFlow, NLP
"""
result = build_full_resume(sample)
print(f"Skills: {len(result.skills)}")
print(f"Experience: {len(result.experience)}")
print(f"Schema v{result.schema_version}")
print(result.model_dump_json(indent=2)[:300])

## Summary: ResumeSchema provides a unified contract. Every engine reads/writes this format.

**A typed schema is the contract that turns eight ad-hoc parsers into one pipeline.**

With `ResumeSchema`, extraction, storage, and matching all speak one shape: Pydantic validates it, defaults keep partial results valid, `schema_version` future-proofs it, and the `confidence`/`source` fields preserve provenance from Ch. 33–37 so downstream scoring can weight by trust. The live build shows the contract working end-to-end — and the placeholder email and empty skills list show exactly where the individual stages still need hardening.

This closes Block F's extraction pipeline. Everything produced here is the input to Ch. 40 — JD Parsing — which parses the *job description* into the same schema shape so resume and JD can finally be matched field by field.